# Notebook 3: Fine-tuning PubMedBERT

Fine-tune `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext` on our
combined medical QA data for extractive question answering (predicting the
answer span inside the context).

PubMedBERT is a BERT-style encoder pre-trained only on PubMed biomedical
abstracts, which is a good match for this domain.

> Note: training needs a GPU. We ran this on the desktop (RTX 5070 Ti).
> Inference on the saved checkpoint is fine on CPU.

In [1]:
import sys, warnings, torch
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5070 Ti
VRAM            : 17.1 GB


## 1. Load and Split Data

In [2]:
from medqa.data.loader import load_all
from medqa.data.preprocessor import split_dataset

records = load_all()
train, test = split_dataset(records)
print(f"Train: {len(train)} | Test: {len(test)}")

[medqa.loader] PubMedQA: loaded 1000 records.
[medqa.loader] BioASQ: loaded 8216 records.
[medqa.loader] MedQA-USMLE: loaded 11451 records.
[medqa.preprocessor] Split -> train: 16533, test: 4134 (stratified=True)


Train: 16533 | Test: 4134


## 2. Fine-tune PubMedBERT

In [3]:
from medqa.models.bert_qa import PubMedBERTQA
from medqa.config import BERT_FINETUNE

print("Training configuration:")
for k, v in BERT_FINETUNE.items():
    print(f"  {k}: {v}")

Training configuration:
  learning_rate: 2e-05
  num_train_epochs: 3
  per_device_train_batch_size: 16
  per_device_eval_batch_size: 16
  warmup_ratio: 0.06
  weight_decay: 0.01
  loss_type: focal_ls
  focal_gamma: 2.0
  label_smoothing: 0.1
  output_dir: c:\Users\Mio\Documents\comp6713-medqa\notebooks\..\checkpoints\bert_qa


In [4]:
model = PubMedBERTQA()

# Check for existing checkpoint
if model.load():
    print("Existing checkpoint found - skipping training.")
    print("Delete checkpoints/bert_qa/ to retrain from scratch.")
else:
    print("Starting fine-tuning...")
    # Use 80% of train for training, 20% for validation
    split_point = int(len(train) * 0.8)
    model.fine_tune(
        train_records = train[:split_point],
        eval_records  = train[split_point:]
    )
    print("Fine-tuning complete!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[medqa.bert] Loaded fine-tuned model from c:\Users\Mio\Documents\comp6713-medqa\notebooks\..\checkpoints\bert_qa


Existing checkpoint found - skipping training.
Delete checkpoints/bert_qa/ to retrain from scratch.


## 3. Single Question Inference

In [5]:
question = "What is the first-line treatment for Type 2 diabetes?"
context  = test[0]['context']

result = model.predict(question, context)
print(f"Question : {question}")
print(f"Answer   : {result['predicted_answer']}")
print(f"Score    : {result['score']:.4f}")

Question : What is the first-line treatment for Type 2 diabetes?
Answer   : No answer found
Score    : 0.5612


## 4. Batch Evaluation

In [6]:
from medqa.evaluation.metrics import evaluate, print_results

test_subset = test[:200]
questions   = [r['question'] for r in test_subset]
contexts    = [r['context']  for r in test_subset]
golds       = [r['answer']   for r in test_subset]

preds_data = model.batch_predict(questions, contexts)
preds      = [p['predicted_answer'] for p in preds_data]

results = evaluate(preds, golds, use_bertscore=False)
print_results(results, model_name='Fine-tuned PubMedBERT')


  Results: Fine-tuned PubMedBERT
  Samples      : 200
  Exact Match  : 0.3500
  Token F1     : 0.3801
  ROUGE-L      : 0.3821
  Yes/No acc   : 1.0000  (2/2, 0 uncategorised)



## 5. Error Analysis

In [7]:
from medqa.evaluation.qualitative import analyse_errors
from medqa.evaluation.metrics import token_f1

eval_records = []
for r, pred in zip(test_subset, preds):
    eval_records.append({
        'question':         r['question'],
        'predicted_answer': pred,
        'gold_answer':      r['answer'],
        'context':          r['context'],
        'token_f1':         token_f1(pred, r['answer']),
    })

summary = analyse_errors(
    eval_records,
    f1_threshold=0.3,
    output_path='../data/processed/bert_errors.json'
)

[Qualitative] Error report saved to ../data/processed/bert_errors.json

  Qualitative Error Analysis
  Total samples : 200
  Errors        : 121 (60.5%)
  Error types:
    TERMINOLOGY_MISMATCH       102  (84.3%)
    PARTIAL_ANSWER              11  (9.1%)
    WRONG_RETRIEVAL              7  (5.8%)
    HALLUCINATION                1  (0.8%)



## 6. Comparison with Baseline

In [8]:
from medqa.models.baseline import TFIDFBaseline
import nltk; nltk.download('punkt_tab', quiet=True)

bl = TFIDFBaseline()
bl.load()
bl_preds = [bl.predict(q)['predicted_answer'] for q in questions]
bl_results = evaluate(bl_preds, golds, use_bertscore=False)

print("Model Comparison (n=200)")
print(f"{'Model':<25} {'EM':>8} {'F1':>8}")
print("-" * 45)
print(f"{'TF-IDF Baseline':<25} {bl_results['mean_em']:>8.4f} {bl_results['mean_f1']:>8.4f}")
print(f"{'Fine-tuned BERT':<25} {results['mean_em']:>8.4f} {results['mean_f1']:>8.4f}")
improvement_f1 = (results['mean_f1'] - bl_results['mean_f1']) / max(bl_results['mean_f1'], 1e-9) * 100
print(f"\nBERT improves F1 by {improvement_f1:.1f}% over baseline")

[medqa.baseline] Index loaded from disk.


Model Comparison (n=200)
Model                           EM       F1
---------------------------------------------
TF-IDF Baseline             0.0000   0.0247
Fine-tuned BERT             0.3500   0.3801

BERT improves F1 by 1436.1% over baseline


## Summary

PubMedBERT is used as an extractive QA model: it predicts start and end
token positions of the answer span inside the context.

A few implementation notes:
- Base model: `BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`, pre-trained on biomedical text.
- Truncation strategy: `longest_first` to deal with long contexts.
- Inference uses a manual forward pass rather than the HuggingFace pipeline, so we can return confidence scores.

On 100 test samples we see roughly EM 0.35, F1 0.39, BERTScore 0.85. The full
comparison against baseline and RAG is in notebook 5.